# FitResultReference

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `analysis`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/FitResultReference.md`


Notebook source link: [FitResultReference.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/FitResultReference.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "FitResultReference"
FAMILY = "analysis"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"FitResultReference: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"FitResultReference: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"FitResultReference: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"FitResultReference: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# FitResultReference: serialize/restore fit metadata and inspect fields.
from nstat.compat.matlab import Analysis, FitResult

dt = 0.02
t = np.arange(0.0, 12.0, dt)
x = np.column_stack([np.sin(2.0 * np.pi * 0.35 * t), np.cos(2.0 * np.pi * 0.15 * t)])
y = rng.poisson(np.exp(-2.0 + 0.9 * x[:, 0] - 0.4 * x[:, 1]) * dt)

fit_native = Analysis.fitGLM(X=x, y=y, fitType="poisson", dt=dt)
fit_native.parameter_labels = ["stim_sin", "stim_cos"]
payload = fit_native.to_structure()
fit = FitResult.fromStructure(payload)

lam_hat = fit.evalLambda(x)
coef = fit.getCoeffs()
param = fit.getParam("intercept")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].bar(np.arange(coef.size), coef, color="tab:blue")
axes[0].set_xticks(np.arange(coef.size), labels=fit.parameter_labels or ["c1", "c2"], rotation=35, ha="right")
axes[0].set_title(f"{TOPIC}: coefficients")
axes[0].set_ylabel("weight")

axes[1].plot(t, lam_hat, color="tab:green", linewidth=1.1)
axes[1].set_title("evalLambda output")
axes[1].set_xlabel("time [s]")
axes[1].set_ylabel("Hz")
plt.tight_layout()
plt.show()

assert np.isfinite(float(param))
assert lam_hat.size == t.size

CHECKPOINT_METRICS = {
    "coef_norm": float(np.linalg.norm(coef)),
    "intercept": float(param),
}
CHECKPOINT_LIMITS = {
    "coef_norm": (0.0, 100.0),
    "intercept": (-20.0, 20.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
